In [0]:
from pyspark.sql.functions import *

df = spark.table("workspace.bronze.bronze_customers")

# Cleaning the df (Selecting imp. fields only)
cleaned_df = df.select(
    col("customer_id"),
    col("customer_name"),
    col("email"),
    col("registration_date"),
    col("country_code"),
    col("channel"),
    col("_modified"),
    col("ingestion_ts") 
).filter(col("customer_id").isNotNull())

In [0]:
missing_values = ['\\n', '?', '', 'null']

# Adding is_valid flag
# Instead of deleting records, Adding missing values to them
cleaned_df = (
    cleaned_df
    .withColumn(
        "is_valid",
        when(
            col("customer_name").isNull() |
            col("email").isNull() |
            col("country_code").isNull() |
            col("channel").isNull() |
            lower(trim(col("customer_name"))).isin(missing_values) |
            lower(trim(col("channel"))).isin(missing_values) |
            lower(trim(col("country_code"))).isin(missing_values) |
            lower(trim(col("email"))).isin(missing_values),
            0
        ).otherwise(1)
    )
    .withColumn(
        "customer_name",
        when(
            col("customer_name").isNull() |
            lower(trim(col("customer_name"))).isin(missing_values),
            None
        ).otherwise(col("customer_name"))
    )
    .withColumn(
        "email",
        when(
            col("email").isNull() |
            lower(trim(col("email"))).isin(missing_values),
            None
        ).otherwise(col("email"))
    )
    .withColumn(
        "channel",
        when(
            col("channel").isNull() |
            lower(trim(col("channel"))).isin(missing_values),
            None
        ).otherwise(col("channel"))
    )
    .withColumn(
        "country_code",
        when(
            col("country_code").isNull() |
            lower(trim(col("country_code"))).isin(missing_values),
            None
        ).otherwise(col("country_code"))
    )
)

In [0]:
# Changing the date format


parsed_date = coalesce(
    try_to_date(col("registration_date"), "MM/dd/yyyy"),
    try_to_date(col("registration_date"), "M/dd/yyyy"),
    try_to_date(col("registration_date"), "MM/d/yyyy"),
    try_to_date(col("registration_date"), "M/d/yyyy"),

    try_to_date(col("registration_date"), "dd/MM/yyyy"),
    try_to_date(col("registration_date"), "d/MM/yyyy"),
    try_to_date(col("registration_date"), "dd/M/yyyy"),
    try_to_date(col("registration_date"), "d/M/yyyy"),

    try_to_date(col("registration_date"), "yyyy/MM/dd"),
    try_to_date(col("registration_date"), "yyyy/M/dd"),
    try_to_date(col("registration_date"), "yyyy/MM/d"),
    try_to_date(col("registration_date"), "yyyy/M/d"),

    try_to_date(col("registration_date"), "MM-dd-yyyy"),
    try_to_date(col("registration_date"), "M-dd-yyyy"),
    try_to_date(col("registration_date"), "MM-d-yyyy"),
    try_to_date(col("registration_date"), "M-d-yyyy"),

    try_to_date(col("registration_date"), "yyyy-MM-dd"),
    try_to_date(col("registration_date"), "yyyy-MM-d"),
    try_to_date(col("registration_date"), "yyyy-M-dd"),
    try_to_date(col("registration_date"), "yyyy-M-d"),

    try_to_date(col("registration_date"), "d-MM-yyyy"),
    try_to_date(col("registration_date"), "dd-M-yyyy"),
    try_to_date(col("registration_date"), "d-M-yyyy")
)

transformed_df = (
    cleaned_df
    .withColumn("parsed_date", parsed_date)
    .withColumn(
        "registration_date",
        when(
            col("parsed_date").isNotNull(),
            date_format(col("parsed_date"), "dd-MM-yyyy")
        ).otherwise(col("registration_date"))
    )
    .drop("parsed_date")
)

display(transformed_df)

In [0]:
transformed_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.silver_customers")